In [ ]:
# 1. Libraries
from pathlib import Path
import h5py
import numpy as np
import matplotlib.pyplot as plt
from detector_io import load_detector_image

# Choose the detector before running the notebook.
detector = "CMOS"                # "CMOS" or "MCP"

def load_image(index):
    return load_detector_image(index, detector, input_folder, image_key)

In [ ]:
# 2. Choose one image and the output PNG
input_folder = Path("raw") / detector
output_folder = Path("output") / detector.lower()
image_index = 16 if detector == "CMOS" else 0
image_key = "data_22"             # CMOS images in these files; data_20 is scalar
output_path = output_folder / "paint_template.png"
mi, ma = 33000, 39000             # manual grayscale limits

# Optional: use the same dark scan as the field-series notebook.
dark_scan_index = None

# MCP integers are zero-based indices; filenames are also accepted by load_image.
# Set scan lists/fields for your MCP acquisition before analyzing a series.
if detector == "MCP":
    for index, path in enumerate(sorted(input_folder.glob("*_TDC_2D_*.txt"))):
        print(index, path.name)


In [ ]:
# 3. Load one image and optionally subtract the dark
image = load_image(image_index)
if dark_scan_index is not None:
    image = image - load_image(dark_scan_index)
if detector == "MCP":
    mi, ma = np.nanpercentile(image, [1, 99.5])  # MCP count scale

In [ ]:
# 4. Show the image
fig, ax = plt.subplots(figsize=(9, 8))
artist = ax.imshow(image, origin="lower", cmap="gray", vmin=mi, vmax=ma)
ax.set(title=f"Scan {image_index}", xlabel="Column (px)", ylabel="Row (px)")
fig.colorbar(artist, ax=ax, label="Intensity")
plt.show()

In [ ]:
# 5. Save original pixels without axes, labels, or resizing.
# Open this PNG in Paint, paint excluded regions RED, and save a NEW file:
# output/cmos/beamstop_mask.png. Do not resize it.
# The analysis notebook reads red pixels and ignores them in all calculations.
output_folder.mkdir(parents=True, exist_ok=True)
plt.imsave(output_path, image, origin="lower", cmap="gray", vmin=mi, vmax=ma)
print("Paint over:", output_path.resolve())

# A separate preview includes axes and colorbar for reference.
# Paint the original-size template above so the mask still matches the data.
fig.savefig(output_folder / "paint_template_preview.png", dpi=150)
